In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_swiss_roll
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path, connected_components

In [3]:
# ---------- (A) Classical MDS / PCoA (final Isomap step) ----------
def classical_mds(D: np.ndarray, n_components: int = 2) -> np.ndarray:
    D = np.asarray(D, dtype=np.float64)
    n = D.shape[0]

    D2 = D ** 2
    J = np.eye(n) - np.ones((n, n)) / n
    B = -0.5 * (J @ D2 @ J)

    # Symmetric eigendecomposition
    eigvals, eigvecs = np.linalg.eigh(B)
    idx = np.argsort(eigvals)[::-1]
    eigvals = eigvals[idx]
    eigvecs = eigvecs[:, idx]

    # Use only components associated with positive eigenvalues
    eigvals = np.maximum(eigvals[:n_components], 0.0)
    X = eigvecs[:, :n_components] * np.sqrt(eigvals + 1e-15)
    return X


# ---------- (B) residual variance (Isomap diagnostic) ----------
def residual_variance(D_geod: np.ndarray, X_embed: np.ndarray) -> float:
    """
    residual variance = 1 - corr(D_geod, D_embed)^2
    where corr is the Pearson correlation between the upper-triangular distance entries.
    """
    D_geod = np.asarray(D_geod, dtype=np.float64)
    n = D_geod.shape[0]

    # Euclidean distances in the embedding
    # Efficient computation based on (x_i - x_j)^2
    # Written explicitly here for clarity:
    diff = X_embed[:, None, :] - X_embed[None, :, :]
    D_emb = np.sqrt(np.sum(diff * diff, axis=2))

    iu = np.triu_indices(n, k=1)
    v1 = D_geod[iu]
    v2 = D_emb[iu]

    # Robust correlation handling
    c = np.corrcoef(v1, v2)[0, 1]
    if not np.isfinite(c):
        return 1.0
    return 1.0 - float(c * c)


# ---------- (C) k-NN graph construction and geodesic distance matrix ----------
def geodesic_distances_knn(X_raw: np.ndarray, n_neighbors: int, ensure_connected: bool = True):
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean")
    nn.fit(X_raw)
    dist, ind = nn.kneighbors(X_raw)  # (n, k)

    n = X_raw.shape[0]
    rows = np.repeat(np.arange(n), n_neighbors)
    cols = ind.reshape(-1)
    data = dist.reshape(-1)

    G = csr_matrix((data, (rows, cols)), shape=(n, n))
    #G = 0.5 * (G + G.T)
    G = G.maximum(G.T)

    if ensure_connected:
        n_comp, labels = connected_components(G, directed=False, return_labels=True)
        if n_comp != 1:
            return None, False  # Disconnected graph

    D = shortest_path(G, directed=False, unweighted=False).astype(np.float64)
    if not np.isfinite(D).all():
        return None, False
    return D, True


# ---------- (D) Final dataset construction ----------
def swissroll_isomap_dataset(
    n_samples: int = 2000,
    noise: float = 0.05,
    random_state: int = 2020,
    # Candidate k grid (can be changed)
    k_values = tuple(range(6, 31, 2)),
    embed_dim: int = 2,
    normalize_targets: bool = True,
    alpha: float = 0.5,
    anchor: int = 0,
    ensure_connected: bool = True,
    verbose: bool = True,
):
    """
    Returns:
      - D_best: (n, n) geodesic distance matrix
      - X_tab: (n, embed_dim) tabular embedding obtained by classical MDS on D_best
      - y_df: dataframe containing the target variables
      - diag_df: diagnostic table used to select k (connectivity and residual variance)
      - k_best: selected k value
      - X_raw: original 3D coordinates, useful for debugging and plotting
    """

    # 1) Generate the 3D Swiss roll and intrinsic coordinate t
    X_raw, t = make_swiss_roll(n_samples=n_samples, noise=noise, random_state=random_state)
    t = t.astype(np.float64)
    h = X_raw[:, 1].astype(np.float64)
    n = X_raw.shape[0]

    if not (0 <= anchor < n):
        raise ValueError(f"anchor must be in [0, {n-1}] but got {anchor}")

    # 2) Sweep over candidate k values
    diag = []
    best = None  # (resvar, k, D, X)
    for k in k_values:
        D, ok = geodesic_distances_knn(X_raw, n_neighbors=int(k), ensure_connected=ensure_connected)
        if not ok:
            diag.append({"k": k, "connected": False, "residual_variance": np.nan})
            continue

        Xk = classical_mds(D, n_components=embed_dim)
        rv = residual_variance(D, Xk)

        diag.append({"k": k, "connected": True, "residual_variance": rv})
        if (best is None) or (rv < best[0]):
            best = (rv, k, D, Xk)

        if verbose:
            print(f"k={k:2d} | residual_variance={rv:.6f}")

    diag_df = pd.DataFrame(diag)

    if best is None:
        raise RuntimeError(
            "No k value produced a connected graph with finite distances. "
            "Increase k_values or reduce noise / n_samples."
        )

    rv_best, k_best, D_best, X_tab = best
    if verbose:
        print(f"\nSelected k_best={k_best} (residual_variance={rv_best:.6f})")

    # 3) Construct the target variables
    y_t = t
    y_h = h
    y_mix = t + alpha * h
    y_anchor = D_best[anchor, :].copy()

    def zscore(v):
        return (v - v.mean()) / (v.std() + 1e-12)

    if normalize_targets:
        y_t = zscore(y_t)
        y_h = zscore(y_h)
        y_mix = zscore(y_mix)
        y_anchor = zscore(y_anchor)

    y_df = pd.DataFrame({
        "id": np.arange(n),
        "y_t": y_t,
        "y_h": y_h,
        "y_mix": y_mix,
        "y_anchor": y_anchor,
    })

    return D_best, X_tab, y_df, diag_df, k_best, X_raw


# ---------- (E) Save generated data ----------
def save_swissroll_bundle(D, X_tab, y_df, diag_df, prefix: str):
    # D
    pd.DataFrame(D, columns=[f"id_{j}" for j in range(D.shape[1])]).to_csv(f"{prefix}_D.csv", index=False)

    # X_tab
    X_df = pd.DataFrame(X_tab, columns=[f"x{j}" for j in range(X_tab.shape[1])])
    X_df.insert(0, "id", np.arange(X_tab.shape[0]))
    X_df.to_csv(f"{prefix}_X.csv", index=False)

    # y
    y_df.to_csv(f"{prefix}_y.csv", index=False)

    # k-selection diagnostics
    diag_df.to_csv(f"{prefix}_k_diagnostics.csv", index=False)

    print("Saved:", f"{prefix}_D.csv")
    print("Saved:", f"{prefix}_X.csv")
    print("Saved:", f"{prefix}_y.csv")
    print("Saved:", f"{prefix}_k_diagnostics.csv")


if __name__ == "__main__":
    D, X_tab, y_df, diag_df, k_best, X_raw = swissroll_isomap_dataset(
        n_samples=1000,
        noise=0.05,
        random_state=2020,
        k_values=tuple(range(6, 31, 2)),
        embed_dim=2,
        normalize_targets=False,
        alpha=0.5,
        anchor=0,
        ensure_connected=True,
        verbose=True
    )

    save_swissroll_bundle(D, X_tab, y_df, diag_df, prefix="swissroll_isomap")
    print("D:", D.shape, "X:", X_tab.shape, "y:", y_df.shape, "k_best:", k_best)

k= 6 | residual_variance=0.002233
k= 8 | residual_variance=0.001063
k=10 | residual_variance=0.000609
k=12 | residual_variance=0.000341
k=14 | residual_variance=0.077056
k=16 | residual_variance=0.074762
k=18 | residual_variance=0.074595
k=20 | residual_variance=0.182733
k=22 | residual_variance=0.212400
k=24 | residual_variance=0.230235
k=26 | residual_variance=0.330945
k=28 | residual_variance=0.383842
k=30 | residual_variance=0.359427

Scelto k_best=12 (residual_variance=0.000341)
Saved: swissroll_isomap_D.csv
Saved: swissroll_isomap_X.csv
Saved: swissroll_isomap_y.csv
Saved: swissroll_isomap_k_diagnostics.csv
D: (1000, 1000) X: (1000, 2) y: (1000, 5) k_best: 12
